# UCI credit-card default: EDA and feature engineering

This notebook is the hands-on companion to the active Flask project. It reads the same raw CSV, uses the same feature-engineering class, and deliberately does **not** train a separate competing model. Run it from the repository root after installing `requirements-dev.txt`.

**Question:** based on six months of historical credit behaviour, can we estimate whether a client will default next month? This is credit-default prediction, not transaction-fraud detection.

## 1. Imports and reproducibility

A fixed random seed makes later train/validation/test examples repeatable. It does not make a result automatically trustworthy.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'raw' / 'UCI_Credit_Card.csv').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.credit_risk.config import DATA_PATH, RAW_FEATURE_COLUMNS, TARGET_COLUMN
from src.credit_risk.features import CreditRiskFeatureEngineer, build_preprocessor

SEED = 42
pd.set_option('display.max_columns', 40)
print(f'Repository root: {ROOT}')
print(f'Data path: {DATA_PATH}')

## 2. Load the source data

Each row is one historical client. `default.payment.next.month` is the target. A target belongs on the answer sheet during training; it must never be supplied as a feature when predicting a new client. `ID` identifies a row and is also excluded from modelling.

In [ ]:
data = pd.read_csv(DATA_PATH)
print('Shape:', data.shape)
display(data.head())
display(data.dtypes.to_frame('dtype').T)
assert set(RAW_FEATURE_COLUMNS + [TARGET_COLUMN, 'ID']) == set(data.columns)

## 3. Data-quality checks

These checks answer basic questions before any model is fitted: are cells missing, are records duplicated, and is `ID` duplicated? For the committed CSV, the expected result is zero for all three checks. That is a property of this file, not evidence that every real credit dataset will be this clean.

In [ ]:
quality = pd.Series({
    'rows': len(data),
    'columns': data.shape[1],
    'missing cells': int(data.isna().sum().sum()),
    'duplicate rows': int(data.duplicated().sum()),
    'duplicate IDs': int(data['ID'].duplicated().sum()),
})
display(quality.to_frame('value'))
display(data.describe().T)

## 4. Target balance

The target is imbalanced: defaults are less common than non-defaults. Accuracy is therefore incomplete. A model that always predicts “no default” would look accurate because it gets the majority class right, while catching no defaults at all.

In [ ]:
target_counts = data[TARGET_COLUMN].value_counts().sort_index()
target_rates = data[TARGET_COLUMN].value_counts(normalize=True).sort_index()
display(pd.DataFrame({'count': target_counts, 'share': target_rates}).rename(index={0: 'no default', 1: 'default'}))
ax = target_counts.rename(index={0: 'No default', 1: 'Default'}).plot.bar(color=['#315ecb', '#0f766e'], rot=0, title='Observed next-month default target')
ax.set_ylabel('Client records')
plt.show()

## 5. Distributions: credit limit and age

A histogram groups values into ranges. The mean is the arithmetic average; the median is the middle observation after sorting. If high values pull the mean above the median, the distribution has a longer high-value tail. This is descriptive evidence, not a reason to delete high limits.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, column, label in [(axes[0], 'LIMIT_BAL', 'Credit limit (NT$)'), (axes[1], 'AGE', 'Age')]:
    axis.hist(data[column], bins=35, color='#315ecb', edgecolor='white')
    axis.axvline(data[column].median(), color='#b45309', linestyle='--', label=f'median = {data[column].median():,.0f}')
    axis.set(title=f'{label} distribution', xlabel=label, ylabel='Records')
    axis.legend()
plt.tight_layout()
plt.show()
print('LIMIT_BAL mean / median:', data['LIMIT_BAL'].mean(), data['LIMIT_BAL'].median())
print('AGE mean / median:', data['AGE'].mean(), data['AGE'].median())

## 6. Repayment behaviour and the target

`PAY_0` is the most recent repayment-status field. Grouping the target by a status code shows the **observed** rate within that historical group. It is an association. It does not prove that a status code alone causes default, and it is not a rule for automatically deciding a person's credit.

In [ ]:
pay0 = data.groupby('PAY_0')[TARGET_COLUMN].agg(records='count', default_rate='mean')
display(pay0)
ax = (pay0['default_rate'] * 100).plot.bar(color='#0f766e', title='Observed default rate by PAY_0')
ax.set(xlabel='PAY_0 source code', ylabel='Default rate (%)')
plt.show()

## 7. Raw linear associations

Pearson correlation measures a linear relationship from −1 to 1. It is useful for a first look, but it cannot identify causes and does not capture every nonlinear interaction that a tree model can learn. The active project computes this same aggregate view for the public page.

In [ ]:
correlations = (data.drop(columns='ID').corr(numeric_only=True)[TARGET_COLUMN].drop(TARGET_COLUMN)
                .sort_values(key=np.abs, ascending=False).head(10))
display(correlations.to_frame('Pearson correlation'))
correlations.sort_values().plot.barh(color=['#b45309' if value < 0 else '#315ecb' for value in correlations.sort_values()])
plt.title('Ten strongest raw linear associations with default')
plt.xlabel('Pearson correlation')
plt.show()

## 8. Reproduce the active feature engineering

The next cell uses `CreditRiskFeatureEngineer` from `src/credit_risk/features.py`, not notebook-only copies. This matters: the website, training program, and notebook should calculate the same columns. Features are past-only summaries: utilisation, six payment ratios, payment-history summaries, averages, trends, and two simple flags.

In [ ]:
X_raw = data[RAW_FEATURE_COLUMNS].copy()
engineer = CreditRiskFeatureEngineer()
X_engineered = engineer.fit_transform(X_raw)
new_columns = [column for column in X_engineered.columns if column not in X_raw.columns]
print('Raw input columns:', len(X_raw.columns))
print('Feature-engineered columns:', len(X_engineered.columns))
display(pd.DataFrame({'new feature': new_columns}))
display(X_engineered[['credit_utilization', 'payment_ratio_1', 'avg_payment_status', 'max_payment_delay', 'months_with_delays', 'avg_bill_amount', 'bill_amount_trend', 'high_utilization']].head())

## 9. Fit preprocessing on training data only

Imputation, scaling, and category encoding learn from data. If they were fitted on the whole CSV before splitting, the validation and test sets would leak small pieces of information into training. The project puts the feature transformer and preprocessor in an sklearn Pipeline so each cross-validation fold fits its own steps correctly.

In [ ]:
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X_raw, data[TARGET_COLUMN], test_size=0.40, stratify=data[TARGET_COLUMN], random_state=SEED
)
preprocessor = build_preprocessor()
X_train_ready = preprocessor.fit_transform(X_train, y_train)
X_holdout_ready = preprocessor.transform(X_holdout)
print('Training rows:', X_train.shape[0], 'Holdout rows:', X_holdout.shape[0])
print('Prepared training matrix:', X_train_ready.shape)
print('Prepared holdout matrix:', X_holdout_ready.shape)
print('Default rate train / holdout:', y_train.mean(), y_holdout.mean())

## 10. Where modelling starts

This notebook stops before fitting a separate model so its results cannot silently diverge from the application. The active training command performs the full reproducible sequence: 60% training, 20% validation, 20% untouched test; stratified cross-validation for logistic regression, random forest, and CatBoost; ROC-AUC model selection; validation F1 threshold choice; then final test reporting.

Run the project commands from the repository root when you want to refresh the official artifacts:

```powershell
python -m src.credit_risk.training --seed 42
python -m src.credit_risk.explain
python -m src.credit_risk.showcase
```

The saved metadata and public website then describe the same active pipeline.